# lora_convert — intermediate checkpoints to loadable adapters (step selection)

Same converter as before, looped over steps 1000/2000/3000 for all three ranks. Step 4000 was already converted and scored on test; these earlier checkpoints allow step selection on a task-aligned metric (argmax GPT-4.1 val Overall) rather than on the diffusion val loss, which did not track subject consistency across the tested checkpoints. Two environment fixes carried over: torchao removed (peft's LoRA dispatch raises on Colab's torchao 0.10), and the transformer is built on CPU rather than a meta device so injection succeeds. `load_state_dict(assign=True)` fills every tensor from the checkpoint, so each adapter is identical to the shipped script's intent.

9 conversions (3 ranks x 3 steps), a few minutes each at ~30 GB peak RAM; ~215 MB of adapters total. Run on a high-RAM or A100 runtime.

## 1. Repo, environment fixes

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's LoRA injection walks every dispatcher, including dispatch_torchao, which raises rather than
# returning False when torchao predates 0.16 (Colab ships 0.10). torchao is unused by this bf16 path;
# removing it makes is_torchao_available() return False so the dispatcher is skipped. Same fix the
# training notebook applies. Guarded so it is a no-op once torchao is already gone / new enough.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

sys.path.insert(0, REPO)
os.chdir(REPO)

# diffusers requires peft>=0.17 to import; the convert cell builds the model on CPU (not a meta
# device) so LoRA injection works under that peft. Report versions and confirm the imports resolve.
import peft, diffusers, transformers
print(f"peft {peft.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")
assert tuple(int(x) for x in peft.__version__.split(".")[:2]) >= (0, 17), \
    "diffusers needs peft>=0.17; do not pin peft below it"
_probe = ("from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available; is_torchao_available();"
          "from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel;"
          "from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline;"
          "print('convert imports OK')")
_chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(_chk.stdout.strip() or _chk.stderr.strip()[-800:])
assert _chk.returncode == 0, "converter imports are broken"

installed.
torchao removed
peft 0.19.1 | diffusers 0.39.0 | transformers 4.51.3
convert imports OK


## 2. Drive, checkpoint inventory

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
EXP   = f"{DRIVE}/lora/experiments"
STEPS = [1000, 2000, 3000]   # 4000 already converted; add the earlier checkpoints for step selection
RANKS = [4, 8, 16]

# the training runs wrote through a symlink at /content/OmniGen2/experiments; the converter
# path arguments below point straight at Drive, so no symlink is required here
for r in RANKS:
    cfg = f"{EXP}/lora_r{r}/lora_r{r}.yml"
    assert os.path.exists(cfg), f"missing config: {cfg}"
    for step in STEPS:
        ckpt = f"{EXP}/lora_r{r}/checkpoint-{step}/pytorch_model_fsdp.bin"
        assert os.path.exists(ckpt), f"missing checkpoint: {ckpt}"
        sz = os.path.getsize(ckpt) / 1e9
        print(f"r{r} step {step}: config + checkpoint present ({sz:.2f} GB)")

Mounted at /content/drive
r4 step 1000: config + checkpoint present (15.88 GB)
r4 step 2000: config + checkpoint present (15.88 GB)
r4 step 3000: config + checkpoint present (15.88 GB)
r8 step 1000: config + checkpoint present (15.89 GB)
r8 step 2000: config + checkpoint present (15.89 GB)
r8 step 3000: config + checkpoint present (15.89 GB)
r16 step 1000: config + checkpoint present (15.91 GB)
r16 step 2000: config + checkpoint present (15.91 GB)
r16 step 3000: config + checkpoint present (15.91 GB)


## 3. Convert — fused checkpoint to adapter

In [ ]:
# The shipped convert_ckpt_to_hf_format.py builds the transformer under init_empty_weights()
# (meta device) before add_adapter. With a current peft that path raises in _create_new_module.
# This reproduces the script's logic exactly, but constructs the module on CPU so the LoRA layers
# materialise normally. load_state_dict(assign=True) then replaces every base+adapter tensor with
# the values from the checkpoint, so the extracted adapter is identical to the script's intent.
import torch, gc
from omegaconf import OmegaConf
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline

TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]

def convert(rank, step):
    cfg  = f"{EXP}/lora_r{rank}/lora_r{rank}.yml"
    ckpt = f"{EXP}/lora_r{rank}/checkpoint-{step}/pytorch_model_fsdp.bin"
    save = f"{EXP}/lora_r{rank}/checkpoint-{step}/transformer_lora"

    conf = OmegaConf.load(cfg)
    assert conf.train.get("lora_ft", False), f"r{rank}: config is not a LoRA run"
    arch_opt = OmegaConf.to_object(conf.model.arch_opt)
    for k, v in arch_opt.items():
        if isinstance(v, list): arch_opt[k] = tuple(v)

    # real CPU module (no init_empty_weights) so peft can inject LoRA on concrete tensors
    transformer = OmniGen2Transformer2DModel(**arch_opt)
    transformer.add_adapter(LoraConfig(
        r=conf.train.lora_rank, lora_alpha=conf.train.lora_rank,
        lora_dropout=conf.train.lora_dropout, init_lora_weights="gaussian",
        target_modules=TARGET_MODULES,
    ))

    state_dict = torch.load(ckpt, mmap=True, weights_only=True)
    missing, unexpect = transformer.load_state_dict(state_dict, assign=True, strict=False)
    # base weights are expected in `missing` only if absent; here the fused checkpoint carries them.
    lora_missing = [k for k in missing if "lora_" in k]
    assert not lora_missing, f"r{rank} step {step}: adapter keys not filled from checkpoint: {lora_missing[:4]}"
    assert not unexpect,     f"r{rank} step {step}: unexpected keys in checkpoint: {unexpect[:4]}"

    adapter = get_peft_model_state_dict(transformer)
    OmniGen2Pipeline.save_lora_weights(save_directory=save, transformer_lora_layers=adapter)
    n = sum(v.numel() for v in adapter.values())
    print(f"r{rank} step {step}: {len(adapter)} adapter tensors, {n/1e6:.3f} M params -> {save}")

    del transformer, state_dict, adapter
    gc.collect()

for r in RANKS:
    for step in STEPS:
        print(f"\n===== converting r{r} step {step} =====")
        convert(r, step)
print("\nall conversions done")

/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.



===== converting r4 step 1000 =====
r4 step 1000: 304 adapter tensors, 2.554 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/lora_r4/checkpoint-1000/transformer_lora

===== converting r4 step 2000 =====
r4 step 2000: 304 adapter tensors, 2.554 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/lora_r4/checkpoint-2000/transformer_lora

===== converting r4 step 3000 =====
r4 step 3000: 304 adapter tensors, 2.554 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/lora_r4/checkpoint-3000/transformer_lora

===== converting r8 step 1000 =====
r8 step 1000: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/lora_r8/checkpoint-1000/transformer_lora

===== converting r8 step 2000 =====
r8 step 2000: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lor

## 4. Verify adapters

In [ ]:
# verify each adapter exists, is small, and reports the expected trainable-parameter count for its rank.
# LoRA params = 2 * rank * sum(in+out over the 4 target projections); within a step the ratio must be exactly 2x from r4 to r8 to r16.
from safetensors.torch import load_file

for step in STEPS:
    prev = None
    print(f"-- step {step} --")
    for r in RANKS:
        path = f"{EXP}/lora_r{r}/checkpoint-{step}/transformer_lora/pytorch_lora_weights.safetensors"
        assert os.path.exists(path), f"adapter not written: {path}"
        sd = load_file(path)
        n_params = sum(v.numel() for v in sd.values())
        ratio = "" if prev is None else f"  ({n_params/prev:.2f}x prev)"
        print(f"  r{r}: {len(sd)} tensors, {n_params/1e6:.3f} M params, {os.path.getsize(path)/1e6:.1f} MB{ratio}")
        prev = n_params

-- step 1000 --
  r4: 304 tensors, 2.554 M params, 10.2 MB
  r8: 304 tensors, 5.107 M params, 20.5 MB  (2.00x prev)
  r16: 304 tensors, 10.214 M params, 40.9 MB  (2.00x prev)
-- step 2000 --
  r4: 304 tensors, 2.554 M params, 10.2 MB
  r8: 304 tensors, 5.107 M params, 20.5 MB  (2.00x prev)
  r16: 304 tensors, 10.214 M params, 40.9 MB  (2.00x prev)
-- step 3000 --
  r4: 304 tensors, 2.554 M params, 10.2 MB
  r8: 304 tensors, 5.107 M params, 20.5 MB  (2.00x prev)
  r16: 304 tensors, 10.214 M params, 40.9 MB  (2.00x prev)
